In [0]:

from sklearn.dummy import DummyClassifier

import pyspark
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
from scipy import stats

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
import xgboost as xgb
from sklearn.neural_network import MLPClassifier

In [0]:
def run_train_test(df,list_classifier,random_state,verbose,k_fold_split):
    int_verbose = int(verbose)
    scaler = MinMaxScaler()
    total_rounds = len(list_classifier)*k_fold_split
    rounds = 1
    list_result = []
    skf = StratifiedKFold(n_splits=k_fold_split,shuffle=True,random_state=random_state)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    for model in list_classifier:
        if not (model != "DT" and model != "XGB" and model != "RF"): 
            min_samples_leaf = int((len(X) * 0.8) *0.1)
            min_samples_split = int(min_samples_leaf*0.1)
            max_depth = 3
            if min_samples_leaf <= 1: min_samples_leaf = 2
            if min_samples_split <= 1: min_samples_split = 2

        match model:
            case "DUM": classifier = DummyClassifier(random_state=random_state,strategy="stratified")
            case "DT": classifier = DecisionTreeClassifier(random_state=random_state,criterion="gini",min_samples_split=min_samples_split,max_depth=max_depth,min_samples_leaf=min_samples_split)
            case "NB": classifier = GaussianNB(priors=None, var_smoothing=1e-09)
            case "KNN": classifier = KNeighborsClassifier(n_neighbors=5,weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None)
            case "XGB": classifier = xgb.XGBClassifier(random_state=random_state,verbosity=int_verbose,objective="binary:logistic",min_child_weight=min_samples_leaf,max_depth=max_depth,eta=0.1,gamma=5)
            case "RF": classifier = RandomForestClassifier(random_state=random_state,verbose=int_verbose,criterion="gini",min_samples_split=min_samples_split,max_depth=max_depth,min_samples_leaf=min_samples_leaf,n_estimators=1000)
            case "MLP": classifier = MLPClassifier(random_state=random_state,verbose=verbose,solver="adam",activation="logistic",max_iter=1000,hidden_layer_sizes=(1,2))
            case "SVM": classifier = svm.SVC(random_state=random_state,verbose=verbose,probability=False,C=1.0, kernel='rbf',degree=3,gamma='scale',coef0=0.0,shrinking=True,tol=0.001,cache_size=200,class_weight=None,max_iter=-1,decision_function_shape='ovr', break_ties=False)

        k_folder_count = 1
        for train_index, test_index in skf.split(X, y):
            print(f"Model: {model}. Cross validation fold[{k_folder_count}] ({(rounds/total_rounds)*100:.2f}%)")

            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            classifier.fit(X_train_scaled,y_train)
            y_train_predicted = classifier.predict(X_train_scaled)
            y_predicted = classifier.predict(X_test_scaled)
            
            # data = []
            data = [
                model
                ,str(accuracy_score(y_train, y_train_predicted)*100)
                ,str(accuracy_score(y_test, y_predicted)*100)
                ,str(precision_score(y_test,y_predicted)*100)
                ,str(recall_score(y_test,y_predicted)*100)
                ,str(f1_score(y_test,y_predicted)*100)
            ]

            list_result.append(data)
            
            # with open(path_txt, mode="a") as file:  file.write(";".join(map(str, data)) + "\n")

            rounds+=1
            k_folder_count+=1
    return(list_result)

In [0]:
def data_to_spark(database,table,result_list):
    dict_column_type = {
        "model": "str"
        ,"accuracy_train": "float"
        ,"accuracy_test": "float"
        ,"presicion": "float"
        ,"recall": "float"
        ,"f1_score": "float"
    }

    df_pd = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in dict_column_type.items()})
    df_pd = pd.DataFrame(result_list,columns=df_pd.columns.to_list())

    spark_session = SparkSession.builder.appName("PandasToPySpark").getOrCreate()
    df_sp = (spark.createDataFrame(df_pd))
    df_sp.write.mode("overwrite").saveAsTable(f"{database}.{table}")
    #spark_session.stop()

In [0]:
def read_parquet():
    path = "/FileStore/tables/dataset/unified_resized.parquet"
    df_sp = spark.read.option("header","true").option("recursiveFileLookup","true").parquet(path)
    df_pd = df_sp.toPandas()
    
    return(df_pd)


In [0]:
def main():
    #list_classifier = ["DUM","DT","NB","KNN","XGB","RF","MLP","SVM"]
    list_classifier = ["DUM","DT","NB","KNN","XGB","RF","MLP"]
    
    df = read_parquet()
    #df_nor = df.query("applied_stimulus == 0").head(1000)
    #df_stm = df.query("applied_stimulus == 1").head(1000)
    #df_limited = pd.concat([df_nor,df_stm],ignore_index=True)

    result_list = run_train_test(df,list_classifier=list_classifier,random_state=42,verbose=False,k_fold_split=5)
    data_to_spark(database="Default",table="Result",result_list=result_list)
    spark.catalog.listTables("Default")


In [0]:
if __name__ == "__main__": main()

Model: DUM. Cross validation fold[1] (2.86%)
Model: DUM. Cross validation fold[2] (5.71%)
Model: DUM. Cross validation fold[3] (8.57%)
Model: DUM. Cross validation fold[4] (11.43%)
Model: DUM. Cross validation fold[5] (14.29%)
Model: DT. Cross validation fold[1] (17.14%)
Model: DT. Cross validation fold[2] (20.00%)
